# Finding the Groups — Clustering, Step by Step

**Runnable twin of [`clustering-walkthrough.html`](clustering-walkthrough.html).** It lets the data vote on its own groupings (no labels), then compares the discovered clusters to our 9 hypothesized archetypes.

**Read the caveat first:** our data is *synthetic* — generated *from* the archetype rules — so clustering will mostly **rediscover** them. That's the point here: it **demonstrates the discovery workflow on data where we know the right answer**. On *real* player data this same code is genuine discovery, and is essential. (See the walkthrough's "Why & when it matters" section.)

**Related:** [Multinomial notebook](multinomial-notebook.ipynb) · [One-vs-Rest notebook](ovr-notebook.ipynb)

---

In [ ]:
# Setup (safe to re-run)
# !pip install pandas scikit-learn matplotlib
import json
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt

CANDIDATES = [Path('../../data/players.json'), Path('data/players.json'), Path('players.json')]
DATA_PATH = next((p for p in CANDIDATES if p.exists()), None)
assert DATA_PATH, 'Could not find players.json — set DATA_PATH manually.'
print('Using:', DATA_PATH.resolve())

## Step 1-2 — Meet the data, and HIDE the answer

**What:** load the same clue columns as the classifier, but set the type label aside — clustering must work without it.

**Why:** the whole point is to see what the clues alone suggest. We keep the true label only to *grade* the clusters at the end (Step 8), never to build them.

In [ ]:
# Steps 1-2 — load clues; keep the true archetype aside as a secret answer key
raw = json.loads(DATA_PATH.read_text(encoding='utf-8'))
players = raw['players'] if isinstance(raw, dict) and 'players' in raw else raw
df = pd.DataFrame(players)

def archetype_from_id(pid):
    n = int(str(pid).split('-')[1])
    if   100 <= n <= 107: return 'new'
    elif 108 <= n <= 141: return 'recreational'
    elif 142 <= n <= 163: return 'regular'
    elif 164 <= n <= 175: return 'grinder'
    elif 176 <= n <= 183: return 'aggressive_predatory'
    elif 184 <= n <= 191: return 'promo_hunter'
    elif 192 <= n <= 197: return 'shared_device_household'
    elif 198 <= n <= 202: return 'cluster_member'
    elif 203 <= n <= 220: return 'healthy_anchor'
    else:                 return 'bot_like'

TRUE_LABEL = df['player_id'].map(archetype_from_id)   # secret answer key (grading only)
FEATURES = [c for c in ['registered_days_ago','lifetime_hands','avg_session_minutes','sessions_last_30d',
            'vpip','pfr','aggression_factor','avg_pot_size_bb','promo_redemptions_30d'] if c in df.columns]
df[FEATURES] = df[FEATURES].fillna(df[FEATURES].median())
print(f'{len(df)} players, {len(FEATURES)} clues. (We secretly know there are {TRUE_LABEL.nunique()} true types.)')

## Step 3 — Put the clues on the same scale

**What:** rescale every clue so distance is fair.

**Why:** clustering groups by distance — without scaling, the biggest-number clue (lifetime_hands) drowns out everything else.

In [ ]:
# Step 3 — standardize (critical for distance-based clustering)
from sklearn.preprocessing import StandardScaler
X = StandardScaler().fit_transform(df[FEATURES])
print('Scaled clue matrix:', X.shape)

## Step 4 — How many groups? (elbow + silhouette)

**What:** try K = 2…15 and score how tidy the groups are. The 'elbow' (where it stops improving) and the silhouette peak suggest a good number.

**Why:** this is where the data tells us whether '9 archetypes' is supported — or whether it prefers fewer/more.

In [ ]:
# Step 4 — elbow (inertia) and silhouette across K
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

Ks = range(2, 16)
inertia, sil = [], []
for k in Ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(X, km.labels_))

fig, ax = plt.subplots(1, 2, figsize=(12,4))
ax[0].plot(list(Ks), inertia, 'o-', color='#b98cff'); ax[0].set_title('Elbow (lower = tidier)'); ax[0].set_xlabel('K')
ax[1].plot(list(Ks), sil, 'o-', color='#33e0c4'); ax[1].set_title('Silhouette (higher = cleaner)'); ax[1].set_xlabel('K')
plt.tight_layout(); plt.show()
print('Silhouette peaks at K =', list(Ks)[int(np.argmax(sil))])

## Step 5 — K-means: sort into hard groups

**What:** fit K-means at our hypothesized K = 9 and see the group sizes.

**Why:** the fast, robust first read — concrete groups we can inspect right away.

In [ ]:
# Step 5 — fit K-means at K=9 (our hypothesis)
K = 9
km = KMeans(n_clusters=K, n_init=10, random_state=42).fit(X)
df['kmeans_cluster'] = km.labels_
print('Players per discovered cluster:')
print(df['kmeans_cluster'].value_counts().sort_index())

## Step 6 — GMM: soft membership + BIC

**What:** fit a Gaussian Mixture Model. It gives each player a *probability* of each group, and BIC gives a principled best-K.

**Why:** captures in-between players, and BIC is a cleaner way to choose K. *(With only ~13 players per group, treat GMM as a second opinion, not gospel.)*

In [ ]:
# Step 6 — GMM: BIC across K, then soft membership for one example player
from sklearn.mixture import GaussianMixture
bic = [GaussianMixture(n_components=k, covariance_type='diag', random_state=42).fit(X).bic(X) for k in Ks]
plt.figure(figsize=(6,3.5)); plt.plot(list(Ks), bic, 'o-', color='#b98cff')
plt.title('GMM BIC (lower = better)'); plt.xlabel('K'); plt.tight_layout(); plt.show()
print('BIC prefers K =', list(Ks)[int(np.argmin(bic))])

gmm = GaussianMixture(n_components=K, covariance_type='diag', random_state=42).fit(X)
probs = gmm.predict_proba(X)
print('\nSoft membership for the first player (chance of each group):')
print(np.round(probs[0], 2))

## Step 7 — Name the clusters

**What:** read each cluster's typical (average) stats and give it a human name.

**Why:** raw cluster numbers mean nothing to a business — a profile makes them real archetypes.

In [ ]:
# Step 7 — average clue values per discovered cluster (read these to name them)
profile = df.groupby('kmeans_cluster')[FEATURES].mean().round(1)
profile

## Step 8 — Compare to our hunches

**What:** line the discovered clusters up against the true archetypes (our secret answer key). A clean grid means each discovered cluster maps to one archetype.

**Why:** the payoff. On real data, a split/merge here means your taxonomy was wrong. **On our synthetic data, expect a near-perfect match — that's the workflow confirming itself** (the data was built from these types).

In [ ]:
# Step 8 — crosstab: discovered cluster (rows) vs true archetype (columns)
ct = pd.crosstab(df['kmeans_cluster'], TRUE_LABEL)
display(ct)

# how cleanly did each discovered cluster land on a single archetype?
purity = ct.max(axis=1).sum() / ct.values.sum()
print(f'\nCluster purity: {purity:.0%}  '
      '(share of players whose cluster is dominated by one true type)')
print('On synthetic data this is high by design — it shows the method recovers the known groups.')

## Steps 9-10 — Lock the taxonomy & hand off

**What:** settle the final archetype list (keep / merge / split based on what we saw), then those named groups become the labels a classifier trains on.

**Why:** the honest full pipeline — *discover the groups, name them, classify into them.* You never assumed the answer; you earned it, then operationalized it.

**Next:** feed the named labels into the **[multinomial](multinomial-notebook.ipynb)** or **[one-vs-rest](ovr-notebook.ipynb)** classifier.

---
### Remember the caveat
Here the clusters matched our 9 types because the data was *built* from them — that validates the **workflow**, not the **taxonomy** (we already knew it). The real value of these exact steps comes with **real player data**, where the clusters could genuinely surprise you. That's why this is a *demonstration* now and a *necessity* later.